In [0]:
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"])
catalog_name = dbutils.widgets.get("catalog")

dbutils.widgets.text("silver_table", "mens_volleyball_clean")

silver_table = dbutils.widgets.get("silver_table")
silver_table_name = f"{catalog_name}.trezio2005_silver.{silver_table}"

Testing the schema enforcement - should throw an error when provided mismatched schema

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

df = spark.table(silver_table_name)

df_new_schema = df.withColumn("New_col", F.lit("test"))

#testing schema enforcement
try:
    df_new_schema.write.format("delta").mode("append").saveAsTable(silver_table_name)
except Exception as e:
    print(f"Schema Enforcement protected data. \nError: {str(e).split(';')[0]}")

Handling the schema evolution scenario

In [0]:

#merging with the new column
(df_new_schema.write 
    .format("delta") 
    .mode("append") 
    .option("mergeSchema", "true") 
    .saveAsTable(silver_table_name)
)

spark.sql(f"SELECT New_col FROM {silver_table_name}").display()

Column mapping - instead of reading and modyfing all the data we use the column mapping funciton to identyfi columns by special ids

In [0]:
#enabling the column mapping function
spark.sql(f"""
            ALTER TABLE {silver_table_name} SET TBLPROPERTIES (
                'delta.columnMapping.mode' = 'name',
                'delta.minReaderVersion' = '2',
                'delta.minWriterVersion' = '5'
            )
          """)

#changing the name of newly added column
spark.sql(f"ALTER TABLE {silver_table_name} RENAME COLUMN New_col TO New_col_renamed")

spark.sql(f"SELECT New_col_renamed FROM {silver_table_name}").display()

In [0]:
#dropping the newly added and renamed column
spark.sql(f"ALTER TABLE {silver_table_name} DROP COLUMN New_col_renamed")

try:
    spark.sql(f"SELECT New_col_renamed FROM {silver_table_name}").display()
except Exception as e:
    print(f"Column has been dropped dropped. \nError: {str(e).split(';')[0]}")

Silent evolution VS Data Contracts

Silent evolution:
- its premise is that the incoming changes are intentional so the new columns are added straight to the table
- enabled by options like: mergeSchema, autoMerge
- works well with pipelines, fast and reliable
- could produce errors if used carelessly

Data Contracts:
- it is a contract between data provider and data consumer. It defines how the data will look. When trying to change its schema it will produce an error
- this can be obtained thanks to functions like: schema enforcement, constraints